# Device functions and subroutines in C++

Until now, we have focused on how to offload data or particular calculations to the GPU.
In most real software applications, however, we will want to pass data through a pipeline of multiple functions and routines before completion.
In this notebook, we will learn how to call functions to run on the device from within existing target regions.
In so doing, we will combine many of the things we have learned up to this point, and let you try porting some code to the GPU using OpenMP yourself.

In the **Device subroutines in C** notebook, we learn about the `declare target` clause that allows us to offload specific functions to the device.
This notebook will look at three further offloading approaches unique to C++; member functions; external functions; and virtual functions.
As such, it is recommended that you work through that previous notebook first before beginning this one.

But first, let's conduct our usual environment checks and move into the C++ direcrtory for this exercise:

In [ ]:
rocm-smi
cd $HOME/AMD_OpenMP_Exercises/2g-OpenMP_device_subroutines/CXX

### How to work through the exercises

This notebook contains three short programming exercises. In each one you will add OpenMP pragmas to a C++ program that currently runs only on the CPU, then check that it still gives the same answer while running on the GPU.

For each exercise:

1. **Read the code first.** The links in each exercise open the source files. Work out which loop should run on the GPU and where the member function is implemented.
2. **Edit the files.** Click a link to open the file, make your changes, and save with Ctrl+S (or Cmd+S on a Mac). The file opens in a new tab; switch back to this notebook's tab when you are done.
3. **Build and run.** Run the `make` cell and then `./bigscience`. The result should match the CPU result printed before you made any changes.
4. **If the compiler complains,** read the message: it names the file and line number. Fix it, save, and run `make` again.
5. **Check that the code really ran on the GPU.** Run `LIBOMPTARGET_INFO=-1 ./bigscience 2>&1 | grep "Launching kernel"`. You should see one line per kernel. If nothing is printed, your code ran on the CPU. (If you use unified shared memory, remember to run `export HSA_XNACK=1` first.)
6. **Compare with our solution** once you have tried it yourself. The solutions are in the folders named in each exercise, and you can run them in the cells that follow.

## C++ member functions

A member function is a function that is declared as part of a class.
Our first example will look at such a function, and how we can offload it to the device.
The example is found in the `1_member_function/0_member_function_portyourself/` directory - let's look at what's there:

In [ ]:
cd $HOME/AMD_OpenMP_Exercises/2g-OpenMP_device_subroutines/CXX/1_member_function/0_member_function_portyourself
make clean
ls

The directory contains two source code files; [`bigscience.cc`](./CXX/1_member_function/0_member_function_portyourself/bigscience.cc) - where `main` is implemented - and [`Science.hh`](./CXX/1_member_function/0_member_function_portyourself/Science.hh), where the `Science` class is defined and its `compute()` **member function** is implemented.
Inspect and understand the code, then compile and run it to understand the baseline CPU behaviour:

In [ ]:
cd $HOME/AMD_OpenMP_Exercises/2g-OpenMP_device_subroutines/CXX/1_member_function/0_member_function_portyourself
make clean
make
export HSA_XNACK=1
./bigscience

### Exercise 1: a member function defined in a header

**Goal:** make the loop in `bigscience.cc` run on the GPU and get the same answer as the CPU run.

**Before you start:** the `compute()` function is implemented in the header [`Science.hh`](./CXX/1_member_function/0_member_function_portyourself/Science.hh), so it is compiled *inline* into the loop. It is already inside the target region, so it needs no extra pragma.

1. Run the compile-and-run cell above once, before changing anything, to see the CPU result.
2. Open [`bigscience.cc`](./CXX/1_member_function/0_member_function_portyourself/bigscience.cc) to edit.
3. Add `#pragma omp target teams loop` immediately before the loop.
4. Choose how the data reaches the GPU:
   - **Unified shared memory** (covered in **Managed memory**): add nothing else. This needs `HSA_XNACK=1`, which the run cell sets.
   - **Explicit memory** (covered in **Explicit memory directive**): add a `map` clause to the pragma for each array the loop uses.
5. Save with Ctrl+S (Cmd+S on a Mac), return to this tab, and run the cell above again. The answer should match the CPU run.

**Compare with our solutions** once you have tried it yourself: [unified shared memory](./CXX/1_member_function/1_member_function/bigscience.cc) and [explicit `map` clauses](./CXX/1_member_function/2_member_function_map/bigscience.cc).

In [ ]:
cd $HOME/AMD_OpenMP_Exercises/2g-OpenMP_device_subroutines/CXX/1_member_function/1_member_function
make clean
make
export HSA_XNACK=1
./bigscience

In [ ]:
cd $HOME/AMD_OpenMP_Exercises/2g-OpenMP_device_subroutines/CXX/1_member_function/2_member_function_map
make clean
make
export HSA_XNACK=0
./bigscience

## External member function

This approach works well when our member function is defined **and** implemented in the header file, since it will be compiled inline.
But what happens when we **declare** a member function in the header, but **implement** it in a separate source code file?

Our second example - in the `2_member_function_external/0_member_function_external_portyourself/` directory - will address this.
Let's look at the contents of the directory:

In [ ]:
cd $HOME/AMD_OpenMP_Exercises/2g-OpenMP_device_subroutines/CXX/2_member_function_external/0_member_function_external_portyourself
make clean
ls

### Exercise 2: a member function implemented in a separate file

**Goal:** port the program again, now that `compute()` is implemented in a different source file from the class that declares it.

The code is laid out as follows:

- [`bigscience.cc`](./CXX/2_member_function_external/0_member_function_external_portyourself/bigscience.cc) contains `main()`.
- [`Science.hh`](./CXX/2_member_function_external/0_member_function_external_portyourself/Science.hh) contains the `Science` class and the **declaration** of the `compute()` member function.
- [`Science_member_functions.cc`](./CXX/2_member_function_external/0_member_function_external_portyourself/Science_member_functions.cc) is new. It contains the **implementation** of `compute()`, so the function is now external to the loop that calls it.

Only the structure of the code has changed since Exercise 1, so the result will be the same.

1. Open [`bigscience.cc`](./CXX/2_member_function_external/0_member_function_external_portyourself/bigscience.cc) and [`Science_member_functions.cc`](./CXX/2_member_function_external/0_member_function_external_portyourself/Science_member_functions.cc) to edit.
2. Repeat the changes from Exercise 1: add `#pragma omp target teams loop` to the loop in `bigscience.cc`.
3. Because `compute()` is no longer inline, the GPU needs its own compiled copy of it. Add a `#pragma omp declare target` directive to the member function in `Science_member_functions.cc`, as in **Device subroutines in C**.
4. Choose how the data reaches the GPU: unified shared memory (needs `HSA_XNACK=1`, which the run cell sets), a `map` clause on the pragma as in Exercise 1, or an unstructured data region.
5. Save each file with Ctrl+S (Cmd+S on a Mac), return to this tab, and run the cell below. The answer should match the CPU result.

**Compare with our solutions** once you have tried it yourself: [unified shared memory](./CXX/2_member_function_external/1_member_function_external/bigscience.cc) and [unstructured data region](./CXX/2_member_function_external/2_member_function_external_data/bigscience.cc).

In [ ]:
cd $HOME/AMD_OpenMP_Exercises/2g-OpenMP_device_subroutines/CXX/2_member_function_external/0_member_function_external_portyourself
make clean
make
export HSA_XNACK=1
./bigscience

In [ ]:
cd $HOME/AMD_OpenMP_Exercises/2g-OpenMP_device_subroutines/CXX/2_member_function_external/1_member_function_external
make clean
make
export HSA_XNACK=1
./bigscience

In [ ]:
cd $HOME/AMD_OpenMP_Exercises/2g-OpenMP_device_subroutines/CXX/2_member_function_external/2_member_function_external_data
make clean
make
export HSA_XNACK=0
./bigscience

## Virtual methods

Our final exercise extends the previous examples with the of use virtual functions. 
Polymorphism aad inheritance are intrinsic aspects of C++, and OpenMP has been written to accomodate this.
This example is contained in the `3_virtual_methods/0_virtual_methods_portyourself/` directory - let's navigate to it and take a look at what's there:

In [ ]:
cd $HOME/AMD_OpenMP_Exercises/2g-OpenMP_device_subroutines/CXX/3_virtual_methods/0_virtual_methods_portyourself
make clean
ls

We now have 4 files;
 - The familiar [`bigscience.cc`](./CXX/3_virtual_methods/0_virtual_methods_portyourself/bigscience.cc) where `main` is implemented,
 - [`Science.hh`](./CXX/3_virtual_methods/0_virtual_methods_portyourself/Science.hh) where the `Science` class is defined, now containing a **virtual function declaration** for the `compute()` function,
 - A new file [`HotScience.hh`](./CXX/3_virtual_methods/0_virtual_methods_portyourself/HotScience.hh) that contains the definition of the `HotScience` class, which inherits from `Science` and defines its own instance of the `compute()` function,
 - and [`HotScience_member_functions.cc`](./CXX/3_virtual_methods/0_virtual_methods_portyourself/HotScience_member_functions.cc) where the `compute()` member function for the `HotScience` class is **implemented**.

You might notice that there has been a minor change in the `compute()` function implementation, so let's compile the code and see the results:

In [ ]:
cd $HOME/AMD_OpenMP_Exercises/2g-OpenMP_device_subroutines/CXX/3_virtual_methods/0_virtual_methods_portyourself
make clean
make
export HSA_XNACK=1
./bigscience

### Exercise 3: virtual functions

**Goal:** port the program one last time, now that `compute()` is a virtual function overridden in a derived class.

Polymorphism and inheritance are central to C++, and OpenMP has been written to accommodate them. For virtual functions, only the **implementation** needs the `#pragma omp declare target` directive. No changes are needed in any header file.

1. Open [`bigscience.cc`](./CXX/3_virtual_methods/0_virtual_methods_portyourself/bigscience.cc) and [`HotScience_member_functions.cc`](./CXX/3_virtual_methods/0_virtual_methods_portyourself/HotScience_member_functions.cc) to edit.
2. Add `#pragma omp target teams loop` to the loop in `bigscience.cc`, as in the previous exercises.
3. Add a `#pragma omp declare target` directive to the implementation in `HotScience_member_functions.cc`.
4. Choose how the data reaches the GPU: unified shared memory (needs `HSA_XNACK=1`, which the run cell sets), `map` clauses on the pragma, or an unstructured data region. The right choice depends on whether the data is reused elsewhere in the program. See **Explicit memory directive** and **Managed memory** for the trade-offs.
5. Save each file with Ctrl+S (Cmd+S on a Mac), return to this tab, and run the cell above. The answer should match the CPU result.

**Compare with our solutions** once you have tried it yourself:

- Unified shared memory: [`bigscience.cc`](./CXX/3_virtual_methods/1_virtual_methods/bigscience.cc) and [`HotScience_member_functions.cc`](./CXX/3_virtual_methods/1_virtual_methods/HotScience_member_functions.cc)
- Explicit `map` clauses: [`bigscience.cc`](./CXX/3_virtual_methods/2_virtual_methods_map/bigscience.cc) and [`HotScience_member_functions.cc`](./CXX/3_virtual_methods/2_virtual_methods_map/HotScience_member_functions.cc)

In [ ]:
cd $HOME/AMD_OpenMP_Exercises/2g-OpenMP_device_subroutines/CXX/3_virtual_methods/1_virtual_methods
make clean
make
export HSA_XNACK=1
./bigscience

In [ ]:
cd $HOME/AMD_OpenMP_Exercises/2g-OpenMP_device_subroutines/CXX/3_virtual_methods/2_virtual_methods_map
make clean
make
export HSA_XNACK=0
./bigscience

Now that you have successfully ported inline, external and virtual member functions to run on the GPU, you can experiment with making the loops larger, changing the logic in `compute()` to be more complex, and testing different memory models.

Throughout this section of the course, we've learnt the fundamentals of OpenMP, and how they can help you run performant code on an AMD GPU or APU.
You should now be equipped to start porting your own code to such architectures.

There follow two optional sections on OpenMP optimisations, and interoperability with HIP.
Feel free to go through them if they sound interesting or useful to you or your work, or continue on to the next section of this course, looking at the HIP programming model, and how dedicated kernels can be used to improve the performance of your code.